# 10. Access within cities, between cities, and against destination density

This notebook estimates the relationship between deprivation and walking
distance to food twice for each definition of a food destination: once over
every census tract of the 92 metropolitan areas together, and once within
metropolitan areas, where only comparisons between tracts of the same area
contribute. Standard errors allow for arbitrary correlation among the tracts of
one area.

It also measures how far the definitions agree on which places are poorly
served, at three levels of comparison; repeats the estimates on the tracts that
the metropolitan boundaries did not cut; and compares the difference in level
between definitions with what the number of destinations alone implies.

The sections are these.

Weighted least squares with clustered standard errors, and the within area
form, obtained by subtracting the population weighted mean of each area from
the outcome and from each characteristic.

The deprivation coefficient under every definition, pooled and within areas,
with its standard error, its t and the sign of each, and the difference between
the two coefficients that the comparison between supermarkets and specialist
fresh sellers rests on.

Rank correlation between pairs of definitions over the 92 area medians, over
all tracts, and within areas. For the within area figure, ranks are positions
between zero and one inside each area, since areas differ by two orders of
magnitude in the number of tracts they hold. The correlation computed
separately inside each area is reported beside it, averaged over areas and over
population, together with its distribution across the 92, and with the partial
correlation that remains once the rank of residential density is removed from
both orderings, since sparse peripheral tracts are far from every kind of
outlet and that alone makes any two distance measures agree.

The same estimates on the tracts at least 99 per cent complete, meaning those
whose blocks all fall inside the boundary of their area.

The distance to the nearest destination that a uniform scattering of
destinations at the observed intensity would produce, which follows from the
square root of the ratio of intensities, set beside the observed ratio.
Intensity is counted as distinct locations as well as registered
establishments, since a public market enters the register once for each of its
stalls at a single coordinate.

The agreement between two definitions on quartiles of need, and the population
subject to two constraints at once, each with the value it would take were the
two things combined unrelated.

Every estimate uses the estimation sample marked in the tract file: the tracts
that carry the marginalisation index, a positive population and density, and a
distance under every definition. Deprivation enters on the scale standardised
over that sample with population weights, and density as the logarithm of
residents per square kilometre.

Inputs are `Results/08_tract_access.parquet`, `Results/08_area_access.parquet`,
`Results/07_block_access.parquet`, `Results/04_tracts.parquet`,
`Results/04_blocks.parquet` and `Results/05_outlets.parquet`.

Output is `Results/10_within_between.parquet`, one row per definition carrying
the pooled and within area coefficients with their standard errors.

Runtime is a few minutes.

## 1. Setup

In [ ]:
import sys, os, importlib

if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

PROJECT = os.environ.get('MXO_BASE', '/content/drive/MyDrive/Mexico_Obesity')
os.environ['MXO_BASE'] = PROJECT
sys.path.insert(0, f'{PROJECT}/Code')
import mxo_utils as mx
importlib.reload(mx)

assert mx.BASE == PROJECT, f'the module resolves to {mx.BASE}'

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

rep = mx.Report('10_within_between').start()
mx.flow_reset()

RESULTS = mx.DIRS['results']

## 2. The tract panel

In [ ]:
TRACT = pd.read_parquet(os.path.join(RESULTS, '08_tract_access.parquet'))
AREA = pd.read_parquet(os.path.join(RESULTS, '08_area_access.parquet'))
OUT = pd.read_parquet(os.path.join(RESULTS, '05_outlets.parquet'))

SETS = [c[len('median_m_'):] for c in TRACT.columns
        if c.startswith('median_m_')]
MCOL = {s: f'median_m_{s}' for s in SETS}
print(f'{len(TRACT):,} tracts, {len(AREA)} areas, {len(SETS)} definitions')

T = TRACT.merge(AREA[['slug', 'tipo']], on='slug', how='left')
NEED = ['imu_depriv_z', 'pop_density_km2', 'macro_region', 'population',
        'slug', 'tipo']
missing = [col for col in NEED if col not in T.columns]
assert not missing, f'the panel lacks {missing}; it carries {sorted(T.columns)}'

T = T[T['estimation_sample'].fillna(False).astype(bool)].copy()
assert T[NEED].notna().all().all(), 'the estimation sample lacks a characteristic'
assert (T.pop_density_km2 > 0).all()
assert T[[MCOL[s] for s in SETS]].notna().all().all(), \
    'a tract in the estimation sample lacks a distance'
T['log_density'] = np.log(T['pop_density_km2'])

print(f'{len(T):,} tracts in the estimation sample, '
      f'{T.population.sum():,.0f} people')
print(f'areas represented: {T.slug.nunique()}')
print(f'tracts an area, median {T.groupby("slug").size().median():.0f}, '
      f'smallest {T.groupby("slug").size().min()}, '
      f'largest {T.groupby("slug").size().max()}')

mx.flow('tracts in the access file', len(TRACT), 'from the aggregation')
mx.flow('tracts in the estimation sample', len(T), 'used here')

## 3. Weighted least squares with clustered standard errors

Each model is fitted by weighted least squares with the population of the tract
as the weight. Standard errors allow for arbitrary correlation among the tracts
of one metropolitan area, since tracts of the same city share a retail
structure and a street network and are not independent observations.

The within city estimate is obtained by subtracting the population weighted
mean of its own area from the outcome and from each characteristic, which
removes everything constant within an area, including the retail system of the
region it belongs to.

In [ ]:
def wls_cluster(y, X, w, cluster, add_const=True, absorbed=0):
    """Coefficients and cluster robust standard errors."""
    if add_const:
        X = np.column_stack([np.ones(len(y)), X])
    XtWX = X.T @ (X * w[:, None])
    beta = np.linalg.solve(XtWX, X.T @ (y * w))
    u = y - X @ beta
    inv = np.linalg.inv(XtWX)

    meat = np.zeros((X.shape[1], X.shape[1]))
    groups = pd.unique(cluster)
    for g in groups:
        m = cluster == g
        score = X[m].T @ (w[m] * u[m])
        meat += np.outer(score, score)

    G, N, K = len(groups), X.shape[0], X.shape[1] + absorbed
    correction = (G / (G - 1)) * ((N - 1) / (N - K))
    V = inv @ meat @ inv * correction

    mean = (w * y).sum() / w.sum()
    r2 = 1 - (w * u ** 2).sum() / (w * (y - mean) ** 2).sum()
    return beta, np.sqrt(np.diag(V)), r2


def demean(frame, cols, by='slug', weight='population'):
    """Each column less the weighted mean of its own group."""
    out = {}
    w = frame[weight]
    for col in cols:
        num = (w * frame[col]).groupby(frame[by]).transform('sum')
        den = w.groupby(frame[by]).transform('sum')
        out[col] = frame[col] - num / den
    return pd.DataFrame(out, index=frame.index)

## 4. The deprivation gradient, pooled and within cities

For every definition the outcome is the logarithm of one plus the distance of
the median resident of the tract. The characteristics are the index of urban
marginalisation, standardised over the estimation sample with population
weights, and the logarithm of residential density. The pooled estimate
adds indicators for the official classification and the macro region. The
within city estimate needs neither, because both are constant inside an area.

In [ ]:
form = pd.get_dummies(T[['tipo', 'macro_region']],
                      drop_first=True).to_numpy(float)
w = T['population'].to_numpy(float)
cluster = T['slug'].to_numpy()
nareas = T.slug.nunique()

rows = []
for s in SETS:
    col = MCOL[s]
    ok = T[col].notna().to_numpy()
    y = np.log1p(T.loc[ok, col].to_numpy(float))
    dep = T.loc[ok, 'imu_depriv_z'].to_numpy(float)
    den = T.loc[ok, 'log_density'].to_numpy(float)

    Xp = np.column_stack([dep, den, form[ok]])
    bp, sp, r2p = wls_cluster(y, Xp, w[ok], cluster[ok])

    # The outcome is demeaned on the logarithmic scale it is modelled on.
    sub = T[ok]
    ylog = pd.Series(y, index=sub.index)
    num = (sub['population'] * ylog).groupby(sub['slug']).transform('sum')
    den_ = sub['population'].groupby(sub['slug']).transform('sum')
    y_within = (ylog - num / den_).to_numpy(float)
    d = demean(sub, ['imu_depriv_z', 'log_density'])
    X_within = np.column_stack([d['imu_depriv_z'].to_numpy(float),
                                d['log_density'].to_numpy(float)])
    bw, sw_, r2w = wls_cluster(y_within, X_within, w[ok], cluster[ok],
                               add_const=False, absorbed=nareas)

    rows.append({'destination set': s, 'tracts': int(ok.sum()),
                 'pooled deprivation': round(float(bp[1]), 4),
                 'pooled se': round(float(sp[1]), 4),
                 'pooled density': round(float(bp[2]), 4),
                 'within deprivation': round(float(bw[0]), 4),
                 'within se': round(float(sw_[0]), 4),
                 'within density': round(float(bw[1]), 4),
                 'pooled r2': round(float(r2p), 4),
                 'within r2': round(float(r2w), 4)})

G = pd.DataFrame(rows).set_index('destination set')
G['pooled t'] = (G['pooled deprivation'] / G['pooled se']).round(2)
G['within t'] = (G['within deprivation'] / G['within se']).round(2)
G['pooled sign'] = np.where(G['pooled deprivation'] > 0, 'further', 'closer')
G['within sign'] = np.where(G['within deprivation'] > 0, 'further', 'closer')
G['sign survives'] = G['pooled sign'] == G['within sign']

print('--- coefficient on deprivation, pooled and within cities ---\n')
print(G[['tracts', 'pooled deprivation', 'pooled se', 'pooled t',
         'within deprivation', 'within se', 'within t',
         'pooled sign', 'within sign', 'sign survives']].to_string())
rep.add_table('deprivation gradient pooled and within cities',
              G.reset_index(), max_rows=12)

print(f'\ndefinitions whose sign survives the move within cities: '
      f'{int(G["sign survives"].sum())} of {len(SETS)}')
print(f'definitions closer where deprived, pooled: '
      f'{int((G["pooled deprivation"] < 0).sum())}')
print(f'definitions closer where deprived, within: '
      f'{int((G["within deprivation"] < 0).sum())}')

In [ ]:
print('--- the contrast the argument rests on ---\n')
PAIR = ['supermarket_club', 'fresh_only']
print(G.loc[PAIR, ['pooled deprivation', 'pooled t', 'within deprivation',
                   'within t', 'sign survives']].to_string())

gap_pooled = (G.loc['supermarket_club', 'pooled deprivation']
              - G.loc['fresh_only', 'pooled deprivation'])
gap_within = (G.loc['supermarket_club', 'within deprivation']
              - G.loc['fresh_only', 'within deprivation'])
print(f'\ndifference between the two coefficients')
print(f'   pooled {gap_pooled:+.4f}')
print(f'   within {gap_within:+.4f}')
print(f'   the within estimate retains {gap_within / gap_pooled:.1%} of it')

## 5. Agreement between definitions, with differences between cities removed

Rank correlation is computed between the distance of the median resident under
each pair of definitions, at three levels. Over the 92 metropolitan areas, it
uses the area medians. Over all tracts, it uses every tract without regard to
area. Within areas, each tract is ranked as a position between zero and one
inside its own area before the correlation is taken over the pooled positions.

Two further columns give the correlation computed separately inside each area,
averaged over the 92 areas and over population. The last gives the partial
rank correlation inside each area, averaged over areas, with the rank of
residential density removed from both orderings. Sparse peripheral tracts are
far from every kind of outlet, so two distance measures agree in part because
both follow density, and the partial correlation is the agreement that remains
once that is taken out.

In [ ]:
def pooled_rho(frame, a, b):
    f = frame[[MCOL[a], MCOL[b]]].dropna()
    return float(spearmanr(f.iloc[:, 0], f.iloc[:, 1])[0])


def within_rho(frame, a, b, by='slug'):
    # Ranks are taken as a position within the area, between zero and one, so
    # that areas holding 27 tracts and areas holding 5,572 contribute on the
    # same scale. A plain rank within each area would carry the size of the
    # area into the pooled correlation.
    f = frame[[by, MCOL[a], MCOL[b]]].dropna()
    r = f.groupby(by)[[MCOL[a], MCOL[b]]].rank(pct=True)
    return float(spearmanr(r.iloc[:, 0], r.iloc[:, 1])[0])


def partial_rho(x, y, z):
    # Spearman correlation of x and y with z partialled out of both, taken on
    # the ranks: the correlation between the parts of the two rankings that
    # the ranking of z does not explain.
    rx, ry, rz = (pd.Series(v).rank().to_numpy(float) for v in (x, y, z))
    rxy = np.corrcoef(rx, ry)[0, 1]
    rxz = np.corrcoef(rx, rz)[0, 1]
    ryz = np.corrcoef(ry, rz)[0, 1]
    den = np.sqrt((1 - rxz ** 2) * (1 - ryz ** 2))
    return float((rxy - rxz * ryz) / den) if den > 0 else np.nan


def per_area_rho(frame, a, b, by='slug', least=4):
    rows = []
    for key, sub in frame.groupby(by):
        f = sub[[MCOL[a], MCOL[b], 'log_density']].dropna()
        if len(f) >= least:
            rows.append({by: key, 'tracts': len(f),
                         'population': float(sub['population'].sum()),
                         'rho': float(spearmanr(f.iloc[:, 0], f.iloc[:, 1])[0]),
                         'rho partial on density': partial_rho(
                             f.iloc[:, 0], f.iloc[:, 1], f.iloc[:, 2])})
    return pd.DataFrame(rows)


PAIRS = [('supermarket_club', 'fresh_only'),
         ('supermarket_club', 'fresh_street'),
         ('supermarket_club', 'fresh_abarrotes'),
         ('supermarket_club', 'all_food'),
         ('supermarket_mini', 'fresh_only'),
         ('fresh_only', 'fresh_abarrotes')]

rows = []
for a, b in PAIRS:
    fa = AREA[[MCOL[a], MCOL[b]]].dropna()
    pa = per_area_rho(T, a, b)
    rows.append({'pair': f'{a} against {b}',
                 'over 92 areas': round(float(
                     spearmanr(fa.iloc[:, 0], fa.iloc[:, 1])[0]), 3),
                 'over all tracts': round(pooled_rho(T, a, b), 3),
                 'within areas, pooled': round(within_rho(T, a, b), 3),
                 'within areas, mean of areas': round(float(pa.rho.mean()), 3),
                 'within areas, population weighted': round(float(
                     (pa.rho * pa.population).sum() / pa.population.sum()), 3),
                 'within areas, partial on density': round(float(
                     pa['rho partial on density'].mean()), 3)})
RHO = pd.DataFrame(rows)
print(RHO.to_string(index=False))
rep.add_table('rank agreement at three levels', RHO, max_rows=12)

PER = per_area_rho(T, 'supermarket_club', 'fresh_only')
print(f'\n--- the same correlation computed inside each area separately ---\n')
print(f'{len(PER)} areas with at least four tracts')
print(PER[['rho', 'rho partial on density']]
      .describe(percentiles=[0.1, 0.5, 0.9]).round(3).to_string())
print(f'\nareas with a negative correlation: '
      f'{int((PER.rho < 0).sum())} of {len(PER)}, '
      f'and {int((PER["rho partial on density"] < 0).sum())} once density is '
      f'removed')
rep.add_table('within area rank correlation, supermarkets against fresh',
              PER.sort_values('rho'), max_rows=92)

## 6. The tracts the boundaries did not cut

A tract crossed by the boundary of a metropolitan area keeps only those of its
blocks whose centres fall inside, so its measures rest on part of its
population. Such tracts are peripheral by construction and differ in character
from the rest, so the results above are repeated on the tracts that are
substantially whole.

In [ ]:
WHOLE = T[T['tract_complete'].fillna(1.0) >= 0.99]
print(f'{len(WHOLE):,} tracts at least 99 per cent complete, '
      f'{WHOLE.population.sum():,.0f} people '
      f'({WHOLE.population.sum() / T.population.sum():.1%})')
print(f'{len(T) - len(WHOLE):,} truncated tracts set aside\n')

formw = pd.get_dummies(WHOLE[['tipo', 'macro_region']],
                       drop_first=True).to_numpy(float)
ww = WHOLE['population'].to_numpy(float)
cw = WHOLE['slug'].to_numpy()
rows = []
for s in SETS:
    col = MCOL[s]
    ok = WHOLE[col].notna().to_numpy()
    y = np.log1p(WHOLE.loc[ok, col].to_numpy(float))
    Xp = np.column_stack([WHOLE.loc[ok, 'imu_depriv_z'].to_numpy(float),
                          WHOLE.loc[ok, 'log_density'].to_numpy(float),
                          formw[ok]])
    bp, sp, _ = wls_cluster(y, Xp, ww[ok], cw[ok])

    sub = WHOLE[ok]
    ylog = pd.Series(y, index=sub.index)
    num = (sub['population'] * ylog).groupby(sub['slug']).transform('sum')
    den_ = sub['population'].groupby(sub['slug']).transform('sum')
    d = demean(sub, ['imu_depriv_z', 'log_density'])
    bw, sw_, _ = wls_cluster((ylog - num / den_).to_numpy(float),
                             d.to_numpy(float), ww[ok], cw[ok],
                             add_const=False, absorbed=sub.slug.nunique())
    rows.append({'destination set': s,
                 'pooled deprivation': round(float(bp[1]), 4),
                 'within deprivation': round(float(bw[0]), 4)})
W = pd.DataFrame(rows).set_index('destination set')
CMP = G[['pooled deprivation', 'within deprivation']].join(
    W, rsuffix=' on whole tracts')
print('--- deprivation coefficient, all tracts against whole tracts ---\n')
print(CMP.to_string())
rep.add_table('deprivation gradient on whole tracts only', CMP.reset_index(),
              max_rows=12)

## 7. What a difference in the number of destinations alone would produce

If destinations were scattered independently at uniform intensity, the typical
distance to the nearest one would fall with the square root of their density.
The ratio of the intensities of two sets therefore predicts a ratio of
distances with no geography entering at all. Comparing that prediction with the
observed ratio separates the part of the difference in level between two
definitions that follows from there being more destinations from the part that
does not.

The intensity is the number of distinct locations rather than the number of
registered establishments. A public market is registered as one record for each
of its stalls, all carrying the market's own coordinate, so a set containing
market trades holds many more records than places a pedestrian can walk to.
Both counts are reported, and the prediction is made from the locations. The
distance to the nearest destination is unaffected by the distinction, because
several establishments at one point are one point to walk to.

In [ ]:
# The intensity of a set is the number of places a pedestrian could walk to,
# not the number of registered establishments. A public market enters the
# register as one record per stall, all at the market's own coordinate, so the
# two differ by a wide margin for the sets that contain market trades.
counts, places = {}, {}
for s in SETS:
    if s not in OUT.columns:
        continue
    sel = OUT.loc[OUT[s].astype(bool), ['latitud', 'longitud']]
    counts[s] = int(len(sel))
    places[s] = int(sel.round(5).drop_duplicates().shape[0])

print('--- registered establishments against distinct locations ---\n')
LOC = pd.DataFrame({'establishments': counts, 'distinct locations': places})
LOC['records per location'] = (LOC['establishments']
                               / LOC['distinct locations']).round(2)
print(LOC.to_string())
rep.add_table('establishments against distinct locations', LOC.reset_index(
    names='destination set'), max_rows=16)
print()

obs = {}
for s in SETS:
    f = T[[MCOL[s], 'population']].dropna().sort_values(MCOL[s])
    cwt = f.population.cumsum() / f.population.sum()
    obs[s] = float(f.loc[cwt >= 0.5, MCOL[s]].iloc[0])

base = 'supermarket_club'
rows = []
for s in SETS:
    if s not in counts:
        continue
    by_record = counts[s] / counts[base]
    by_place = places[s] / places[base]
    rows.append({'destination set': s,
                 'distinct locations': places[s],
                 'location ratio': round(by_place, 2),
                 'predicted from locations': round(float(np.sqrt(by_place)), 2),
                 'predicted from records': round(float(np.sqrt(by_record)), 2),
                 'observed distance ratio': round(obs[base] / obs[s], 2)})
DN = pd.DataFrame(rows).set_index('destination set')
DN['observed over predicted'] = (DN['observed distance ratio']
                                 / DN['predicted from locations']).round(2)
print('--- level differences against what destination density predicts ---\n')
print(DN.to_string())
rep.add_table('observed level differences against a uniform intensity '
              'prediction', DN.reset_index(), max_rows=12)
print('\nSets whose members are blocks rather than establishments are absent '
      'from this table, since no establishment count exists for them.')

## 8. Two statistics against the benchmark of independence

Two quantities are reported with the value each would take were the two things
combined unrelated. The first is the number of areas falling in the same
quartile of need under two definitions, against the number expected from the
marginal totals, with the standard deviation of that expectation. The second is
the share of population beyond one mile from a supermarket and in a tract where
most dwellings have no car, against the product of the two shares separately.

In [ ]:
A = AREA.dropna(subset=[MCOL['supermarket_club'], MCOL['fresh_only']]).copy()
A['q_sup'] = pd.qcut(A[MCOL['supermarket_club']].rank(ascending=False), 4,
                     labels=range(1, 5))
A['q_fresh'] = pd.qcut(A[MCOL['fresh_only']].rank(ascending=False), 4,
                       labels=range(1, 5))
grid = pd.crosstab(A['q_sup'], A['q_fresh'])
same = int(np.trace(grid.values))
n = len(A)
rowm = grid.sum(axis=1).to_numpy()
colm = grid.sum(axis=0).to_numpy()
expected = float((rowm * colm / n).sum())
variance = float(sum((rowm[i] * colm[i] / n) * (1 - rowm[i] * colm[i] / n / n)
                     for i in range(len(rowm))))
sd = float(np.sqrt(variance))
print(f'areas in the same quartile of need under both definitions {same} of {n}')
print(f'expected if the orderings were independent              {expected:.1f}')
print(f'standard deviation of that expectation                  {sd:.2f}')
print(f'the observed count lies                                 '
      f'{(same - expected) / sd:+.2f} standard deviations from it')
print('\nagreement is therefore no better than chance, and not '
      'distinguishable from chance in either direction.')

BL = pd.read_parquet(os.path.join(RESULTS, '07_block_access.parquet'),
                     columns=['CVEGEO_MZA', 'POBTOT',
                              'distance_m_supermarket_club'])
TRv = pd.read_parquet(os.path.join(RESULTS, '04_tracts.parquet'),
                      columns=['CVEGEO', 'veh_share'])
B4 = pd.read_parquet(os.path.join(RESULTS, '04_blocks.parquet'),
                     columns=['CVEGEO_MZA', 'CVEGEO'])
D = (BL.merge(B4, on='CVEGEO_MZA').merge(TRv, on='CVEGEO')
       .dropna(subset=['distance_m_supermarket_club', 'veh_share']))
far = D['distance_m_supermarket_club'] > 1609
carless = (1 - D['veh_share']) > 0.5
tot = D.POBTOT.sum()
p_far = D.POBTOT.where(far, 0).sum() / tot
p_car = D.POBTOT.where(carless, 0).sum() / tot
p_both = D.POBTOT.where(far & carless, 0).sum() / tot
print(f'\nbeyond one mile from a supermarket                       {p_far:.4f}')
print(f'in a tract where most dwellings have no car             {p_car:.4f}')
print(f'both                                                    {p_both:.4f}')
print(f'expected if the two were independent                    '
      f'{p_far * p_car:.4f}')
print(f'observed over independent                               '
      f'{p_both / (p_far * p_car):.2f}')
BENCH = pd.DataFrame([
    {'statistic': 'areas in the same quartile of need', 'observed': same,
     'expected if independent': round(expected, 1),
     'observed over expected': round(same / expected, 2)},
    {'statistic': 'share of population under both constraints',
     'observed': round(p_both, 4),
     'expected if independent': round(p_far * p_car, 4),
     'observed over expected': round(p_both / (p_far * p_car), 2)}])
rep.add_table('two statistics against the benchmark of independence', BENCH,
              max_rows=10)

## 9. Checks

Each check names the condition under which it fails.

In [ ]:
print('--- checks ---\n')

d = int(T['CVEGEO'].duplicated().sum())
print(f'1. duplicate tracts                          {d}')
assert d == 0

# Removing what is constant inside an area cannot leave a mean inside an area.
chk = demean(T, ['imu_depriv_z'])
resid = ((T.population * chk['imu_depriv_z']).groupby(T.slug).sum()
         / T.population.groupby(T.slug).sum()).abs().max()
print(f'2. largest group mean left after demeaning   {resid:.2e}')
assert resid < 1e-8

# A standard error has to be positive and finite.
bad = int(((G['pooled se'] <= 0) | (G['within se'] <= 0)
           | ~np.isfinite(G['pooled se']) | ~np.isfinite(G['within se'])).sum())
print(f'3. standard errors not positive and finite   {bad}')
assert bad == 0

# A coefficient of determination lies between zero and one.
bad = int(((G['pooled r2'] < 0) | (G['pooled r2'] > 1)
           | (G['within r2'] < 0) | (G['within r2'] > 1)).sum())
print(f'4. explained shares outside zero to one     {bad}')
assert bad == 0

# A rank correlation lies between minus one and one.
for col in RHO.columns:
    if col != 'pair':
        assert RHO[col].between(-1, 1).all(), col
print('5. every rank correlation inside minus one to one  yes')

# Pooling positions within an area must agree with averaging over areas. A
# large gap between them would mean the pooled figure is carrying something
# other than agreement inside areas.
gap = (RHO['within areas, pooled']
       - RHO['within areas, population weighted']).abs().max()
print(f'6. pooled against population weighted, largest gap {gap:.3f}')
assert gap < 0.2, RHO.to_string(index=False)

# Deprivation has weighted mean zero and unit weighted spread on the sample.
wm = float((T.population * T.imu_depriv_z).sum() / T.population.sum())
ws = float(np.sqrt((T.population * (T.imu_depriv_z - wm) ** 2).sum()
                   / T.population.sum()))
print(f'   deprivation on the sample, weighted mean {wm:.2e}, weighted sd {ws:.6f}')
assert abs(wm) < 1e-6 and abs(ws - 1) < 1e-6

print(f'\n7. areas contributing to the within estimate {nareas}')
print(f'8. sign of the deprivation gradient survives within cities, '
      f'{int(G["sign survives"].sum())} of {len(SETS)} definitions')
print(f'9. supermarkets, pooled {G.loc["supermarket_club", "pooled deprivation"]:+.4f}'
      f' and within {G.loc["supermarket_club", "within deprivation"]:+.4f}')
print(f'10. specialist fresh, pooled {G.loc["fresh_only", "pooled deprivation"]:+.4f}'
      f' and within {G.loc["fresh_only", "within deprivation"]:+.4f}')
print(f'11. the contrast between them retains '
      f'{gap_within / gap_pooled:.1%} of its pooled size')
print(f'12. agreement between supermarkets and fresh inside areas, '
      f'mean over areas {PER.rho.mean():+.3f}, '
      f'{int((PER.rho < 0).sum())} of {len(PER)} negative')

## 10. Write results

In [ ]:
out = os.path.join(RESULTS, '10_within_between.parquet')
G.reset_index().to_parquet(out, index=False)
print('written:', out)

print()
print(mx.flow_table().to_string(index=False))
mx.save_flow(os.path.join(mx.DIRS['logs'], '10_sample_flow.csv'))
rep.add_table('attrition', mx.flow_table(), max_rows=20)

rep.note(
    f'within metropolitan areas the deprivation coefficient is '
    f'{G.loc["supermarket_club", "within deprivation"]:+.4f} for supermarkets '
    f'with clubs and {G.loc["fresh_only", "within deprivation"]:+.4f} for '
    f'specialist fresh and staple sellers, against '
    f'{G.loc["supermarket_club", "pooled deprivation"]:+.4f} and '
    f'{G.loc["fresh_only", "pooled deprivation"]:+.4f} pooled; the sign is '
    f'unchanged for {int(G["sign survives"].sum())} of {len(SETS)} definitions; '
    f'rank agreement between the two is '
    f'{RHO.iloc[0]["over 92 areas"]:+.3f} over the 92 areas and '
    f'{PER.rho.mean():+.3f} averaged over the correlations computed inside '
    f'each area')
rep.save()